<a href="https://course.bioinfo.se/open.html?nb=session02-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#007ACC;color:#FFFFFF;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in VS Code</a>

# Session 2 lab: working like a computational biologist {.unnumbered}

This lab continues from the Session 2 page's mini challenge, with less help
at each step:

- **Part A, fill in:** complete marked lines in code that otherwise
  works: explore the Becares RNA-seq table, plot it, test a relationship,
  read a sequence from NCBI, and finish a small class of your own.
- **Part B, adapt:** change working code so that it answers a new
  question, and fix a cell with three bugs.
- **Part C, solve:** one open challenge with a specification and hints
  only, a twelve-globin analysis. Three more challenges (download a gene
  and plot it, explore the table with seaborn, a statistical test) follow
  as optional extras, which are not part of the quiz.

In Part A, every `todo("...")` call marks code for you to write: replace the whole
`todo(...)` call with your code. Work through the cells **in order, top
to bottom**. Until you fill it in, a cell stops with
`NotImplementedError: TODO in this cell: ...`, which tells you what is
missing. That is expected.

Several sections fetch records from NCBI, so you need an internet
connection. Run the notebook in VS Code on a Colab machine (Google Colab extension;
see the Setup page). Answer the Session 2 lab quiz on Canvas with **your own
notebook's output**.

**Before submitting or sharing a notebook: Restart Kernel → Run All.**

In [ ]:
# Install the libraries Colab does not come with (does nothing if they are already installed)
import importlib.util, subprocess, sys
for module, package in [("Bio", "biopython")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from Bio import Entrez, SeqIO
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction

Entrez.email = "your.name@student.su.se"    # replace with your own address: NCBI requires one

URL = "https://raw.githubusercontent.com/JoachimGoedhart/VolcaNoseR/master/Becares-diffgenes_HFHC.csv"

def todo(what):
    '''Placeholder for code you write: replace the whole todo(...) call with your own code.'''
    raise NotImplementedError(f"TODO in this cell: {what}")

# Part A: Fill in

## A1. Load and inspect the table

The same dataset as on the Session 2 page: mouse liver RNA-seq,
LXRα-S196A mutant against wild-type mice, both on a high-fat/high-cholesterol
diet (Becares et al. 2019).
This time, look at the column `lfcSE`: the **standard error** of each
log2 fold change, i.e. how uncertain the estimate is.

In [ ]:
df = pd.read_csv(URL)
print(df.shape)
print(df.head())

median_se = todo('the median of the lfcSE column')
max_padj = todo('the largest adjusted p-value (column padj) in the file')
print(f"median lfcSE:          {median_se:.3f}")
print(f"largest adjusted p:    {max_padj:.3f}")

## A2. Filter

A stricter threshold than the page's: adjusted p-value below 0.01.

In [ ]:
strict = todo('the rows of df with padj below 0.01')
n_up = todo('how many of those genes have a positive log2_FoldChange')
print(f"genes with padj < 0.01: {len(strict)}  (up: {n_up}, down: {len(strict) - n_up})")

## A3. Plot first, test second

Is the uncertainty of a fold change (`lfcSE`) related to how highly the
gene is expressed (`baseMean`)? Plot it first, with expression on a log
axis; then test.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
todo("scatter plot of lfcSE (y) against baseMean (x), small points")
ax.set_xscale("log")
ax.set_xlabel("mean expression (log scale)"); ax.set_ylabel("lfcSE (standard error of log2 FC)")
plt.tight_layout(); plt.show()

r_raw, p_raw = stats.pearsonr(df["baseMean"], df["lfcSE"])
print(f"Pearson, raw baseMean:  r = {r_raw:.2f}, p = {p_raw:.2g}")
rho, p_rho = todo('Spearman correlation between baseMean and lfcSE')
print(f"Spearman:               rho = {rho:.2f}, p = {p_rho:.2g}")

## A4. A mystery sequence

A 30-nucleotide fragment of a human gene. Which gene is revealed in A5.

In [ ]:
mystery = Seq("CGCAGGGAGCGGAGCGAGGAGCTGTCCGAG")

gc = todo('the GC fraction of mystery')
rev_comp = todo('the reverse complement of mystery')
protein = todo('mystery translated to protein')
print("GC fraction:        ", round(gc, 2))
print("reverse complement: ", rev_comp)
print("translated:         ", protein)

## A5. From NCBI to a local file and back

Fetch the full record for accession `NM_134268.5`, a human
globin-family gene, inside a `with` block. Then save it as a local FASTA
file and read it back, again with `with`.

In [ ]:
with todo("open an Entrez.efetch handle: database nuccore, id NM_134268.5, rettype gb, retmode text") as handle:
    record = SeqIO.read(handle, "genbank")
print("description:", record.description)
print("length (nt):", len(record.seq))

with open("mystery_gene.fasta", "w") as out:
    SeqIO.write(record, out, "fasta")
with open("mystery_gene.fasta") as handle:
    local = todo('read the FASTA file back as one record (SeqIO.read)')
print("read back from the local file:", local.id, len(local.seq))

position = todo('the 0-based position of the mystery fragment in the mRNA (str.find)')
print("the mystery fragment starts at position", position)

## A6. Your own class

The Session 2 page's `Protein` class bundles a sequence with the methods
that act on it. Finish a similar class for an mRNA record: `__init__`
stores the data as attributes, and each method uses `self`, the object
it was called on. Then make an object from the record you fetched in A5.

In [ ]:
class Transcript:
    """An mRNA record from NCBI: the gene, the mRNA and its coding sequence."""

    def __init__(self, gene, record):
        self.gene = gene
        self.mrna = todo("store the record's sequence (record.seq) as the attribute mrna")
        cds_feature = [f for f in record.features if f.type == "CDS"][0]
        self.cds = cds_feature.extract(record.seq)

    def mrna_length(self):
        return len(self.mrna)

    def coding_fraction(self):
        """The fraction of the mRNA that is coding sequence."""
        return todo("the length of self.cds divided by the mRNA length (use the method above)")

    def protein(self):
        """The protein: the coding sequence translated up to the stop codon."""
        return todo("self.cds translated, stopping at the first stop codon (to_stop=True)")


cygb = Transcript("CYGB", record)
print("gene:           ", cygb.gene)
print("mRNA length:    ", cygb.mrna_length(), "nt")
print("coding fraction:", round(cygb.coding_fraction(), 3))
print("protein length: ", len(cygb.protein()), "aa")

# Part B: Adapt

In Part B the code already works: your job is to **change** it so it
answers a new question. There are no `todo(...)` lines; edit the code
itself.

## B1. Adapt a query

The cell below finds the gene that went **up** the most among the genes
with padj < 0.01 (your table `strict` from A2). Change it so that it
finds the gene that went **down** the most.

In [ ]:
# ADAPT: change this cell so it finds the most DOWN-regulated gene instead
most_up = strict.sort_values("log2_FoldChange", ascending=False).iloc[0]
print(f"most up-regulated gene with padj < 0.01: {most_up['Gene']} "
      f"(log2 FC {most_up['log2_FoldChange']:.2f})")

## B2. Fix three bugs

The cell below should compute two things:

1. the average GC fraction of three sequence fragments, weighting each
   fragment by its length, and
2. the number of genes in the table with a log2 fold change above 1.

It contains three bugs. One stops the cell with an `ImportError`, one
with a `KeyError`, and one lets the cell run but gives a wrong answer.
Use the routine from the Session 2 page (read the error, inspect,
documentation, verify) to fix all three. **Check that your answer is
plausible:** a weighted average must lie between the smallest and the
largest of the values it averages.

In [ ]:
from Bio.SeqUtils import GC

fragments = ["ATGGTGCATCTGACTCCTGAG", "CGCAGGGAGCGGAGCGAGGAGCTG", "ATGGTGCTGTCTCCTGCCGACAAGACC"]
gc_values = [GC(f) for f in fragments]
lengths = [len(f) for f in fragments]
weights = lengths * 2                  # weight each fragment by (twice) its length
weighted_gc = sum(g * w for g, w in zip(gc_values, weights)) / sum(weights)
print("GC per fragment:     ", [round(g, 3) for g in gc_values])
print("length-weighted GC:  ", round(weighted_gc, 3))

n_big = (df["log2FoldChange"] > 1).sum()
print("genes with log2 FC > 1:", n_big)

## B3. Adapt the mini challenge to new genes

The function below is the Session 2 page's mini-challenge code, packaged so
that it works for any set of accessions, and it runs on the page's six
globins. Adapt it in two ways:

1. add a column `untranslated` to each row: the mRNA length minus the
   length of the coding sequence (`len(cds.extract(record.seq))`);
2. run it on six more human globins from the alpha- and beta-globin
   gene clusters instead:

| Gene | Protein | RefSeq accession |
|---|---|---|
| *HBG1* | gamma-globin 1 (fetal) | `NM_000559.3` |
| *HBG2* | gamma-globin 2 (fetal) | `NM_000184.3` |
| *HBD* | delta-globin | `NM_000519.4` |
| *HBE1* | epsilon-globin (embryonic) | `NM_005330.4` |
| *HBZ* | zeta-globin (embryonic) | `NM_005332.3` |
| *HBQ1* | theta-globin | `NM_005331.5` |

In [ ]:
# ADAPT: add an "untranslated" column, and run the function on the six new globins above
def globin_table(accessions):
    '''One row per gene: fetch all records in ONE request, then read length, GC and protein length.'''
    with Entrez.efetch(db="nuccore", id=",".join(accessions.values()), rettype="gb", retmode="text") as handle:
        records = list(SeqIO.parse(handle, "genbank"))
    rows = []
    for gene, record in zip(accessions, records):
        cds = [f for f in record.features if f.type == "CDS"][0]
        rows.append({"gene": gene, "accession": record.id, "mrna_length": len(record.seq),
                     "gc": round(gc_fraction(record.seq), 3),
                     "protein_length": len(cds.qualifiers["translation"][0])})
    table = pd.DataFrame(rows)
    assert list(table["accession"]) == list(accessions.values()), "records came back in a different order"
    return table

page_accessions = {"HBB": "NM_000518.5", "HBA1": "NM_000558.5", "HBA2": "NM_000517.6",
                   "MB": "NM_001382809.1", "NGB": "NM_021257.4", "CYGB": "NM_134268.5"}
globin_table(page_accessions)

# Part C: Solve

One open challenge, with a specification and some hints, and an empty
cell: the code is yours. There is more than one good solution.
Use what you learned in Parts A and B and on the Session 2 page, the library
documentation, and an AI assistant if the course rules allow it, but
**check every result** (plot first, test second, and ask whether the
answer is plausible). The challenge builds on your adapted function from
B3. Three more challenges follow as optional extras, which are not part
of the quiz.

## C. Twelve globins: does a longer mRNA mean a longer protein?

Combine the page's six globins with the six new ones from B3, and:

1. build one DataFrame with twelve rows, with a column `group` ("page"
   or "new") and a column `coding_fraction` (the fraction of the mRNA
   that is coding sequence);
2. **plot first:** protein length against mRNA length, one colour per
   group, each point labelled by its gene. Before computing anything,
   look at the plot. Does any point stand out? Which one, and what might
   make it different, biologically or technically?
3. compute the Pearson correlation between mRNA length and protein
   length over all twelve genes, and again without the point you
   identified, and compare the two;
4. explain in a few sentences what the plot shows about mRNAs and
   proteins, and which result you would report.

**About removing points.** Don't remove a point because it makes the
correlation weaker, or stronger. An influential observation deserves a
closer look, but you need a *reason* to exclude it (for example, a
wrong record or a different kind of molecule). If there is no such
reason, keep it: report both analyses and explain why the result
changes.

Hints: your adapted `globin_table` from B3 does most of the work;
`pd.concat([a, b])` stacks two DataFrames; seaborn's `hue=` colours by a
column; `ax.annotate` labels a point.

In [ ]:
# C: your code here (table, correlations, labelled plot)

*C, your explanation (two or three sentences): replace this line.*

# Optional extras (not part of the quiz)

Three more open challenges, for those who want more practice. They are
not part of the quiz. Each starts from scratch.

## Extra 1. Download a gene and look at it

1. Use `Entrez.esearch` to find the RefSeq mRNA of the human leptin
   gene, *LEP*: search the `nuccore` database for
   `LEP[Gene Name] AND Homo sapiens[Organism] AND refseq_select[filter] AND biomol_mrna[PROP]`.
   Then fetch the record with `Entrez.efetch` inside a `with` block.
2. Report its accession, mRNA length, GC fraction, the length of the
   protein (from the CDS feature) and the fraction of the mRNA that is
   coding sequence.
3. Plot the GC fraction in a sliding window along the mRNA (for example
   100 nt windows, every 10 nt), and mark where the CDS is. Is the coding
   region different from the rest?
4. **Think about it (for discussion, not graded):** why can an mRNA that
   encodes a protein of *n* amino acids be much longer than 3*n*
   nucleotides? Which parts of your leptin mRNA are not translated, and
   how long are they?
5. Optional: repeat for a gene of your own choice.

Hints: `Entrez.read(handle)` turns an esearch result into a dictionary
whose `"IdList"` holds the record IDs; `cds.location.start` and
`.end` give the CDS coordinates; `ax.axvspan(start, end, alpha=0.2)`
shades a region.

In [ ]:
# Extra 1: your code here (search, fetch, report, sliding-window plot)

## Extra 2. Explore the RNA-seq table with seaborn

Make three plots of the Becares table (`df`), each followed by one
sentence (in a markdown cell) saying what it shows:

1. a **joint plot** (`sns.jointplot`) of log10 mean expression against
   log2 fold change: a scatter (or hexbin) plot with the two
   distributions on its margins;
2. a **heatmap** (`sns.heatmap`) of the Spearman correlation matrix of
   the numeric columns (`df.select_dtypes("number").corr(method="spearman")`),
   with the values written in the cells (`annot=True`) and a colour
   scale from -1 to 1;
3. one plot of your own choice that shows something the first two
   don't.

Hint: add a column first, e.g. `df["log10_baseMean"] = np.log10(df["baseMean"])`.

In [ ]:
# Extra 2: your plots here

*Extra 2, one sentence per plot: replace this line.*

## Extra 3. A statistical question: plot first, test second

Among the significant genes (padj < 0.05), are the genes that went
**up** expressed at a different level (`baseMean`) than those that went
**down**?

1. Plot the two groups first (a box plot, violin plot or histogram, on a
   log scale).
2. Choose a suitable test. Expression values are very skewed, so a
   rank-based test is a good choice: the Mann-Whitney U test
   (`stats.mannwhitneyu(a, b, alternative="two-sided")`).
3. Report the number of genes and the median `baseMean` in each group,
   the test statistic and the p-value, and interpret the result in two
   sentences. Does the test agree with the plot?
4. Optional: ask and test a question of your own about the table.

In [ ]:
# Extra 3: your plot and test here

*Extra 3, your interpretation (two sentences): replace this line.*

## Done

When every cell runs, answer the Session 2 lab quiz on Canvas with your own
output, and paste your written answers from Part C into the quiz. The
optional extras are not part of the quiz.